# BEFORE YOU RUN THIS FILE:

## To run the OpenAI portion of the code, you will need an OpenAI API key.
## You can either use your own OpenAI API key or be added to our project and request an API key through our project.
Please email Tongyu at guotq@bc.edu
## Once you have your API key, proceed with the following steps in our class JupyterLab environment.
## Assume that you have already gone to our GitHub page and copied or cloned the repository.

# Step 1
Open the JupyterLab Terminal.

# Step 2
Type the following command and press Enter:
touch .env

# Step 3
Then type the following command and press Enter:
nano .env

# Step 4
The previous step opens the .env file in the terminal editor. Then type the following:
OPENAI_API_KEY=your_OpenAI_API_key_here

# Step 5
After entering the API key, press: Ctrl + O
Then press Enter to save the file.
Next, press: Ctrl + X to exit the editor.

# Step 6
Once this is done, run the file. If you encounter any issues, copy the codebook into your main parent folder and run it from there.

## Imports

In [1]:
import os
import zipfile
import pickle
import warnings
from pathlib import Path
from datetime import datetime
import numpy as np                     # numpy>=1.26
import pandas as pd                    # pandas>=2.0
import matplotlib.pyplot as plt        # matplotlib>=3.8
import seaborn as sns                  # seaborn>=0.13
%pip install pyarrow lightgbm dotenv openai
import pyarrow


import lightgbm as lgb                 # lightgbm>=4.0
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
)                                      # scikit-learn>=1.4

warnings.filterwarnings("ignore")
print(f"numpy={np.__version__}  pandas={pd.__version__}  lightgbm={lgb.__version__}")


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
numpy=2.2.6  pandas=2.3.3  lightgbm=4.6.0


In [2]:
import os, subprocess

REPO_URL = "https://github.com/tongyuguo/HelpHerInvest.git"
REPO_DIR = Path("HelpHerInvest")
data_dir = REPO_DIR / "Data"

def clone_or_pull():
    if (REPO_DIR / ".git").is_dir():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"])
    else:
        subprocess.run(["git", "clone", REPO_URL])

clone_or_pull()

Already up to date.


## Configuration


In [3]:
# PATHS
DATA_ZIP_PATH   = data_dir / "final_dataset_20260224v2.csv.zip"
DATA_CSV_NAME   = "final_dataset_20260224v2.csv"   

ARTIFACTS_DIR   = REPO_DIR / "artifacts"
MODEL_PATH      = ARTIFACTS_DIR / "lgbm_model.pkl"
TRAIN_DATA_PATH = ARTIFACTS_DIR / "train_ranked.parquet"
VAL_DATA_PATH   = ARTIFACTS_DIR / "val_ranked.parquet"
TEST_DATA_PATH  = ARTIFACTS_DIR / "test_ranked.parquet"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# SPLIT FRACTIONS
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
TEST_FRAC  = 0.15

# TARGET COLUMN
TARGET_COL = "y"           # 1 if fwd_excess > 0
DATE_COL   = "Date"
TICKER_COL = "Ticker"

# COLUMNS TO EXCLUDE FROM FEATURES 
NON_FEATURE_COLS = [DATE_COL, TICKER_COL, "fwd_excess", "fwd_return", "y", "fwd_rank", "target"]

# FEATURE COLUMNS
RANK_COLS = [
    "mom_1m", "mom_3m", "mom_6m", "mom_12m", "mom_12m_ex_1m",
    "rel_3m_spy", "rel_6m_spy", "rel_12m_spy",
    "vol_3m", "vol_6m",
    "drawdown_6m", "drawdown_12m",
    "pct_above_200dma",
]

#  LIGHTGBM HYPERPARAMETERS
LGBM_PARAMS = dict(
    n_estimators  = 1000,
    learning_rate = 0.01,
    max_depth     = 12,
    num_leaves    = 31,
    class_weight  = "balanced",
    random_state  = 42,
)


## Data Loading

In [4]:
def load_data(zip_path: Path, csv_name: str) -> pd.DataFrame:

    with zipfile.ZipFile(zip_path) as z:
        with z.open(csv_name) as f:
            df = pd.read_csv(f)

    df[DATE_COL] = pd.to_datetime(df[DATE_COL])  
    tickers = df[TICKER_COL].unique()
    print(f"Loaded  : {df.shape[0]} rows  x  {df.shape[1]} cols")
    print(f"Tickers : {len(tickers)}")
    print(f"Date range: {df[DATE_COL].min().date()}  →  {df[DATE_COL].max().date()}")
    return df


def save_data(df: pd.DataFrame, path: Path, fmt: str = "parquet") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if fmt == "parquet":
        df.to_parquet(path, index=False)
    elif fmt == "csv":
        df.to_csv(path, index=False)
    elif fmt == "pickle":
        df.to_pickle(path)
    else:
        raise ValueError(f"Unsupported fmt={fmt!r}. Choose parquet | csv | pickle.")
    print(f"Saved  {df.shape} → {path}")


def load_saved_data(path: Path) -> pd.DataFrame:
    ext = path.suffix.lower()
    if ext == ".parquet":
        df = pd.read_parquet(path)
    elif ext == ".csv":
        df = pd.read_csv(path)
    elif ext in (".pkl", ".pickle"):
        df = pd.read_pickle(path)
    else:
        raise ValueError(f"Unknown extension {ext}. Supported: .parquet, .csv, .pkl")
    print(f"Loaded {df.shape} ← {path}")
    return df


df_raw = load_data(DATA_ZIP_PATH, DATA_CSV_NAME)


Loaded  : 302024 rows  x  18 cols
Tickers : 1993
Date range: 2010-02-28  →  2026-02-28


## Preprocessing


In [5]:
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    n_before = len(df)
    df_clean = df.dropna().copy()
    n_after  = len(df_clean)
    print(f"Dropped {n_before - n_after:,} rows with NaNs  ({n_before:,} → {n_after:,})")

    # Binary target
    df_clean["y"] = (df_clean["fwd_excess"] > 0).astype(int)
    pos = df_clean["y"].sum()
    print(f"Class balance  →  positive: {pos}  ({pos/n_after:.1%})  |  negative: {n_after-pos}  ({(n_after-pos)/n_after:.1%})")

    # Log price feature
    df_clean["log_adj_close"] = np.log(df_clean["adj_close"])

    return df_clean


df_clean = preprocess_data(df_raw)


Dropped 27,513 rows with NaNs  (302,024 → 274,511)
Class balance  →  positive: 132974  (48.4%)  |  negative: 141537  (51.6%)


## Feature Engineering — Cross-Sectional Ranking

In [6]:
def apply_cross_sectional_rank(df: pd.DataFrame, rank_cols: list, date_col: str = DATE_COL) -> pd.DataFrame:
    df_rank = df.copy()
    for col in rank_cols:
        if col not in df_rank.columns:
            print(f"  Warning: column {col!r} not found — skipping.")
            continue
        df_rank[col] = df_rank.groupby(date_col)[col].rank(pct=True)

    # Quintile label for the forward return (0 = worst, 4 = best)
    df_rank["fwd_rank"] = df_rank.groupby(date_col)["fwd_excess"].rank(pct=True)
    df_rank["target"]   = pd.cut(
        df_rank["fwd_rank"],
        bins=[0, .2, .4, .6, .8, 1.0],
        labels=[0, 1, 2, 3, 4],
    )
    return df_rank


def retrieve_features(df: pd.DataFrame, exclude_cols: list = NON_FEATURE_COLS) -> list:
    feature_cols = sorted([c for c in df.columns if c not in exclude_cols])
    print(f"Feature count: {len(feature_cols)}")
    return feature_cols



df_rank = apply_cross_sectional_rank(df_clean, RANK_COLS)
FEATURE_COLS = retrieve_features(df_rank)
print("Sample features:", FEATURE_COLS[:10])


Feature count: 15
Sample features: ['adj_close', 'drawdown_12m', 'drawdown_6m', 'log_adj_close', 'mom_12m', 'mom_12m_ex_1m', 'mom_1m', 'mom_3m', 'mom_6m', 'pct_above_200dma']


## Train / Val / Test Split

In [7]:
def time_split(
    df: pd.DataFrame,
    date_col: str  = DATE_COL,
    train_frac: float = TRAIN_FRAC,
    val_frac:   float = VAL_FRAC,
    test_frac:  float = TEST_FRAC,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if abs(train_frac + val_frac + test_frac - 1.0) > 1e-8:
        raise ValueError("Fractions must sum to 1.0")

    unique_dates = df[date_col].drop_duplicates().sort_values().reset_index(drop=True)
    n = len(unique_dates)
    i_train = int(n * train_frac)
    i_val   = int(n * (train_frac + val_frac))

    train_df = df[df[date_col].isin(unique_dates.iloc[:i_train])].copy()
    val_df   = df[df[date_col].isin(unique_dates.iloc[i_train:i_val])].copy()
    test_df  = df[df[date_col].isin(unique_dates.iloc[i_val:])].copy()

    for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
        print(f"{name:5s}: {len(split):>7,} rows  |  "
              f"{split[date_col].min().date()}  →  {split[date_col].max().date()}")
    return train_df, val_df, test_df



train_df, val_df, test_df = time_split(df_rank)

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET_COL]
X_val,   y_val   = val_df[FEATURE_COLS],   val_df[TARGET_COL]
X_test,  y_test  = test_df[FEATURE_COLS],  test_df[TARGET_COL]


Train: 176,701 rows  |  2011-01-31  →  2021-05-31
Val  :  47,488 rows  |  2021-06-30  →  2023-08-31
Test :  50,322 rows  |  2023-09-30  →  2025-11-30


## Model Training

In [8]:
def train_lgbm(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val:   pd.DataFrame,
    y_val:   pd.Series,
    params:  dict = LGBM_PARAMS,
) -> lgb.LGBMClassifier:
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
    )
    print(f"Training complete — best iteration: {model.best_iteration_}")
    return model

lgb_model = train_lgbm(X_train, y_train, X_val, y_val)


[LightGBM] [Info] Number of positive: 86755, number of negative: 89946
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.016449 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3825
[LightGBM] [Info] Number of data points in the train set: 176701, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Training complete — best iteration: 0


## Save & Load Model

In [9]:
def save_model(model, path: Path) -> None:
    """
    Serialize the model to disk using pickle.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(model, f, protocol=pickle.HIGHEST_PROTOCOL)
    size_kb = path.stat().st_size / 1024
    print(f"Model saved  → {path}  ({size_kb:.1f} KB)  |  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


def load_model(path: Path):
    """
    Load the pickled model from disk.
    """
    with open(path, "rb") as f:
        model = pickle.load(f)
    print(f"Model loaded ← {path}")
    return model


# Save the model 
save_model(lgb_model, MODEL_PATH)

# Reload and verify 
lgb_model_loaded = load_model(MODEL_PATH)


Model saved  → HelpHerInvest/artifacts/lgbm_model.pkl  (3455.5 KB)  |  2026-04-30 18:12:20
Model loaded ← HelpHerInvest/artifacts/lgbm_model.pkl


# NLP Ticker Selection

In [10]:
# create unique ticker dataframe 

def create_unique_ticker_df(df):
    if "Ticker" not in df.columns:
        raise ValueError("Column 'Ticker' not found in dataframe.")
    
    unique_ticker = (
        df["Ticker"]
        .dropna()
        .astype(str)
        .str.upper()
        .str.strip()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
        .to_frame(name="Ticker")
    )
    
    return unique_ticker

unique_ticker = create_unique_ticker_df(df_rank)

print(unique_ticker.head())
print("Number of unique tickers:", len(unique_ticker))

  Ticker
0      A
1     AA
2    AAL
3   AAON
4    AAP
Number of unique tickers: 1913


In [11]:
from pprint import pprint

#API 
from dotenv import load_dotenv
import os
load_dotenv() 

from openai import OpenAI
import pandas as pd
import io
import json

client = OpenAI()

def get_30_unique_tickers(unique_ticker: pd.DataFrame, user_input: str, model: str = "gpt-5-mini") -> pd.DataFrame:

    if "Ticker" not in unique_ticker.columns:
        raise ValueError("unique_ticker must contain a 'Ticker' column.")

    # Clean ticker column
    valid_tickers = (
        unique_ticker["Ticker"]
        .dropna()
        .astype(str)
        .str.upper()
        .str.strip()
        .drop_duplicates()
    )

    valid_ticker_set = set(valid_tickers)

    # Convert dataframe to in-memory CSV
    ticker_buffer = io.BytesIO()
    valid_tickers.to_frame(name="Ticker").to_csv(ticker_buffer, index=False)
    ticker_buffer.seek(0)
    ticker_buffer.name = "unique_ticker.csv"

    # Upload in-memory file
    uploaded_file = client.files.create(file=ticker_buffer, purpose="user_data")

    # Call model
    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "developer",
                "content": [
                    {
                        "type": "input_text",
                        "text": (
                            "You are selecting stock tickers for a downstream model.\n"
                            "You must use the attached CSV file as the only allowed universe of tickers.\n"
                            "Based on the user's interest, return exactly 30 unique ticker symbols.\n"
                            "Rules:\n"
                            "- Only return tickers that exist in the attached file.\n"
                            "- Return exactly 30 unique tickers.\n"
                            "- Do not include explanations.\n"
                            "- Do not include markdown.\n"
                            '- Return valid JSON in this exact format: {"tickers": ["AAPL", "MSFT", "..."]}'
                        )
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {"type": "input_file", "file_id": uploaded_file.id},
                    {"type": "input_text", "text": user_input}
                ]
            }
        ]
    )

    # Parse JSON output
    try:
        result = json.loads(response.output_text)
        tickers = result["tickers"]
    except Exception as e:
        raise ValueError(f"Model output was not valid JSON: {response.output_text}") from e

    # Clean + validate output
    cleaned = []
    seen = set()

    for t in tickers:
        t = str(t).upper().strip()
        if t in valid_ticker_set and t not in seen:
            cleaned.append(t)
            seen.add(t)

    if len(cleaned) != 30:
        raise ValueError(
            f"Expected 30 valid unique tickers, but got {len(cleaned)} after validation.\n"
            f"Raw model output: {response.output_text}"
        )

    # Return dataframe
    selected_tickers_df = pd.DataFrame({"Ticker": cleaned})
    return selected_tickers_df

In [12]:
user_input = "I like the food industry."

selected_tickers_df = get_30_unique_tickers(unique_ticker, user_input)

print(selected_tickers_df)

   Ticker
0     ADM
1     BUD
2     CAG
3    CAKE
4    COST
5    COKE
6      KO
7     CPB
8     KHC
9     GIS
10    KDP
11    HSY
12    CMG
13    DRI
14    EAT
15   CAVA
16    DPZ
17   BF-A
18    HRL
19     BJ
20   FIZZ
21   AMZN
22   AGCO
23     DE
24     CF
25   CASY
26    GLW
27    CLX
28    KMB
29    CPK


##  Rank Selected Tickers

In [ ]:
def rank_selected_tickers(
    selected_tickers_df: pd.DataFrame,
    feature_df: pd.DataFrame,
    model,
    feature_cols: list,
    ticker_col: str = "Ticker",
    date_col: str = "Date",
    as_of_date=None,
) -> pd.DataFrame:
    if ticker_col not in selected_tickers_df.columns:
        raise ValueError(f"selected_tickers_df must contain '{ticker_col}'")

    missing_features = [c for c in feature_cols if c not in feature_df.columns]
    if missing_features:
        raise ValueError(f"feature_df is missing required feature columns: {missing_features}")

    if ticker_col not in feature_df.columns or date_col not in feature_df.columns:
        raise ValueError(f"feature_df must contain '{ticker_col}' and '{date_col}'")

    # Clean tickers in both dfs
    selected = selected_tickers_df.copy()
    selected[ticker_col] = (
        selected[ticker_col].astype(str).str.upper().str.strip()
    )

    df = feature_df.copy()
    df[ticker_col] = df[ticker_col].astype(str).str.upper().str.strip()
    df[date_col] = pd.to_datetime(df[date_col])

    # Determine scoring cutoff date
    if as_of_date is None:
        as_of_date = df[date_col].max()
    else:
        as_of_date = pd.to_datetime(as_of_date)

    # Keep only rows up to the as-of date
    df = df[df[date_col] <= as_of_date].copy()

    # Keep only selected tickers
    selected_set = set(selected[ticker_col])
    df = df[df[ticker_col].isin(selected_set)].copy()

    if df.empty:
        raise ValueError("No matching ticker rows found in feature_df for the selected tickers.")

    # For each ticker, use the latest available row up to as_of_date
    df = df.sort_values([ticker_col, date_col])
    latest_rows = df.groupby(ticker_col, as_index=False).tail(1).copy()

    # Drop rows with missing model features
    before = len(latest_rows)
    latest_rows = latest_rows.dropna(subset=feature_cols).copy()
    dropped = before - len(latest_rows)
    if dropped > 0:
        print(f"Dropped {dropped} selected tickers due to missing feature values.")

    if latest_rows.empty:
        raise ValueError("No selected tickers remain after dropping missing feature rows.")

    # Score with model
    X_score = latest_rows[feature_cols]
    latest_rows["score"] = model.predict_proba(X_score)[:, 1]

    # Rank descending
    ranked = latest_rows.sort_values("score", ascending=False).reset_index(drop=True)
    ranked["rank"] = ranked["score"].rank(method="first", ascending=False).astype(int)

    # Select output columns
    output_cols = [ticker_col, date_col, "score", "rank"]
    return ranked[output_cols]

In [14]:
ranked_tickers_df = rank_selected_tickers(
    selected_tickers_df=selected_tickers_df,
    feature_df=df_rank,
    model=lgb_model,
    feature_cols=FEATURE_COLS,
    ticker_col="Ticker",
    date_col="Date",
    as_of_date=None,   # uses latest available date
)

print(ranked_tickers_df.head(30))

print("\nTop 12 recommended tickers:")
print(ranked_tickers_df["Ticker"].unique()[:12])

   Ticker       Date     score  rank
0     KMB 2025-11-30  0.529404     1
1     HRL 2025-11-30  0.521555     2
2      BJ 2025-11-30  0.515662     3
3     GLW 2025-11-30  0.515193     4
4     KDP 2025-11-30  0.511879     5
5    FIZZ 2025-11-30  0.511272     6
6     DRI 2025-11-30  0.510748     7
7    AGCO 2025-11-30  0.508622     8
8    CAKE 2025-11-30  0.503568     9
9     ADM 2025-11-30  0.503308    10
10    BUD 2025-11-30  0.502872    11
11   AMZN 2025-11-30  0.501392    12
12    CLX 2025-11-30  0.500538    13
13    CPB 2025-11-30  0.500241    14
14    DPZ 2025-11-30  0.498206    15
15     CF 2025-11-30  0.497018    16
16    HSY 2025-11-30  0.493437    17
17   CASY 2025-11-30  0.487985    18
18   BF-A 2025-11-30  0.487387    19
19   COKE 2025-11-30  0.486214    20
20    CPK 2025-11-30  0.479609    21
21    KHC 2025-11-30  0.476631    22
22    CMG 2025-11-30  0.475766    23
23    CAG 2025-11-30  0.475213    24
24   CAVA 2025-11-30  0.471600    25
25    GIS 2025-11-30  0.465214    26
2